In [58]:
import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split 

import time

import sklearn.metrics as skm

import matplotlib.pyplot as plt

In [59]:
# Transform to turn into tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081,)) # mean y std
])

# Load the dataset and normalize it
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

# Load the test dataset
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=False,
    transform=transform
)

In [60]:
# DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=1028,
    num_workers=4,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=2056,
    num_workers=4,
    shuffle=True,
    drop_last=True
)

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

Entrenando en: cuda


In [62]:
def createTheMNIST():

    class mnistNet(nn.Module):
        def __init__(self):
            super().__init__()

            # input
            self.input = nn.Linear(784,64)

            # hidden layer
            self.fc1 = nn.Linear(64,32)
            self.fc2 = nn.Linear(32,32)

            # output
            self.output = nn.Linear(32,10)

        def forward(self,x):
            x = torch.flatten(x, start_dim=1)
            
            x = F.relu(self.input(x))
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            return self.output(x)

    net = mnistNet()

    lossfun = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(net.parameters(), lr=0.1)

    return net, lossfun, optimizer

In [63]:
def function2trainTheModel():

    timerInFunction = time.process_time()

    numepochs = 10

    net, lossfun, optimizer = createTheMNIST()

    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []

    net.train()
    for epochi in range(numepochs):

        batchAcc = []
        batchLoss = []
        for X, y in train_loader:
            
            yHat = net(X)
            loss = lossfun(yHat, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batchLoss.append(loss.item())

            matches = torch.argmax(yHat, axis=1) == y
            matchesNumeric = matches.float()
            accuracyPct = 100*torch.mean(matchesNumeric)
            batchAcc.append( accuracyPct )

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        net.eval()
        X,y = next(iter(test_loader))
        with torch.no_grad():
            yHat = net(X)

        testAcc.append( 100*torch.mean((torch.argmax(yHat,axis=1)==y).float()) )

        comptime = time.process_time() - timerInFunction
        print(f'Epoch {epochi+1}/{numepochs}, elapsed time: {comptime:.2f} sec, test accuracy: {np.mean(testAcc)}, train accuracy: {np.mean(trainAcc)}')

    return trainAcc, testAcc, losses, net

In [64]:
trainAcc, testAcc, losses, net = function2trainTheModel()

Epoch 1/10, elapsed time: 2.71 sec, test accuracy: 20.136186599731445, train accuracy: 18.551254272460938
Epoch 2/10, elapsed time: 5.69 sec, test accuracy: 24.97568130493164, train accuracy: 21.967662811279297
Epoch 3/10, elapsed time: 9.19 sec, test accuracy: 26.086252212524414, train accuracy: 24.022764205932617
Epoch 4/10, elapsed time: 13.26 sec, test accuracy: 27.88180923461914, train accuracy: 25.558917999267578
Epoch 5/10, elapsed time: 16.29 sec, test accuracy: 30.116729736328125, train accuracy: 27.618408203125
Epoch 6/10, elapsed time: 20.98 sec, test accuracy: 29.98540687561035, train accuracy: 28.26881980895996
Epoch 7/10, elapsed time: 25.52 sec, test accuracy: 31.343801498413086, train accuracy: 28.750886917114258
Epoch 8/10, elapsed time: 29.18 sec, test accuracy: 33.086090087890625, train accuracy: 30.430068969726562
Epoch 9/10, elapsed time: 32.09 sec, test accuracy: 34.84111404418945, train accuracy: 32.085514068603516
Epoch 10/10, elapsed time: 36.07 sec, test accur

In [65]:
timerOutsideFunction = time.process_time()

for i in range(10):
    function2trainTheModel() 

TotalExperimentTime = time.process_time() - timerOutsideFunction
print(f"Total elapsed experiment time: {TotalExperimentTime/60:.2f} minutes")

Epoch 1/10, elapsed time: 5.79 sec, test accuracy: 18.28793716430664, train accuracy: 11.674827575683594
Epoch 2/10, elapsed time: 9.63 sec, test accuracy: 20.865758895874023, train accuracy: 16.95877456665039
Epoch 3/10, elapsed time: 12.97 sec, test accuracy: 23.37872886657715, train accuracy: 20.75786018371582
Epoch 4/10, elapsed time: 16.47 sec, test accuracy: 25.59581756591797, train accuracy: 23.23099708557129
Epoch 5/10, elapsed time: 21.38 sec, test accuracy: 27.908559799194336, train accuracy: 25.864416122436523
Epoch 6/10, elapsed time: 25.10 sec, test accuracy: 30.001623153686523, train accuracy: 28.053298950195312
Epoch 7/10, elapsed time: 29.71 sec, test accuracy: 31.545303344726562, train accuracy: 29.770368576049805
Epoch 8/10, elapsed time: 32.89 sec, test accuracy: 32.57538986206055, train accuracy: 31.171592712402344
Epoch 9/10, elapsed time: 36.11 sec, test accuracy: 33.70082092285156, train accuracy: 32.27372741699219
Epoch 10/10, elapsed time: 40.70 sec, test accur